# LFW — 00. 공통 Aligned 112×112 Crop 생성

InsightFace `buffalo_l` (RetinaFace)로 얼굴을 검출하고 ArcFace 표준
5-point similarity transformation으로 112×112 RGB uint8 crop을 생성합니다.

검출 실패 표본은 center crop으로 대체하지 않고 별도 실패 manifest에 기록합니다.
이 산출물은 모든 Step 2 모델(ArcFace, AdaFace, MagFace)의 공통 입력입니다.

**참고 논문:**
- Deng et al., "ArcFace: Additive Angular Margin Loss for Deep Face Recognition", CVPR 2019
- Deng et al., "RetinaFace: Single-shot Multi-level Face Localisation in the Wild", CVPR 2020


In [1]:
# cell 1 : 환경 설정 및 프로젝트 루트 탐색
from __future__ import annotations

import logging
from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(C:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
)
logger = logging.getLogger("aligned_crop_materializer")

EXECUTE_STAGE = True   # 실제 생성 시에만 True
OVERWRITE = True       # 기존 산출물 덮어쓰기 금지


In [2]:
# cell 2 : 입출력 경로 및 설정 파라미터
import pandas as pd

# 입력
SOURCE_MANIFEST_PATH = (
    PROJECT_ROOT / CONFIG["datasets"]["lfw"]["manifest_path"]
)

# 출력 (YAML의 datasets.lfw.aligned_crops에서 읽음)
LFW_ALIGNED_CONFIG = CONFIG["datasets"]["lfw"]["aligned_crops"]
OUTPUT_DIR = (
    PROJECT_ROOT / LFW_ALIGNED_CONFIG["manifest_path"]
).parent

# 검출기 설정
DETECTOR_NAME = "buffalo_l"
DETECTION_SIZE = tuple(CONFIG["aligned_crops"]["image_size"])  # (112, 112) → 검출은 (640, 640)
DETECTION_INPUT_SIZE = (640, 640)
# PROVIDERS = ("CPUExecutionProvider",)  # GPU 사용 시 ("CUDAExecutionProvider",)
# cell 2 : 입출력 경로 및 설정 파라미터
PROVIDERS = ("CUDAExecutionProvider", "CPUExecutionProvider")

logger.info("SOURCE_MANIFEST_PATH: %s", SOURCE_MANIFEST_PATH)
logger.info("OUTPUT_DIR: %s", OUTPUT_DIR)

if not SOURCE_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"LFW face_manifest.csv가 없습니다. "
        f"notebooks/lfw/00_data_preparation.ipynb를 먼저 실행하세요: "
        f"{SOURCE_MANIFEST_PATH}"
    )
source_manifest = pd.read_csv(SOURCE_MANIFEST_PATH)
logger.info("source manifest: %d행", len(source_manifest))
source_manifest.head()


2026-07-25 03:12:32,192 [INFO] aligned_crop_materializer — SOURCE_MANIFEST_PATH: C:\ronbun\data\interim\lfw\face_manifest.csv
2026-07-25 03:12:32,193 [INFO] aligned_crop_materializer — OUTPUT_DIR: C:\ronbun\data\interim\common\aligned_112\lfw
2026-07-25 03:12:32,231 [INFO] aligned_crop_materializer — source manifest: 13233행


,image_id,identity_id,split,image_path
0,lfw:Aaron_Pena:Aaron_Pena_0001,lfw:Aaron_Pena,calibration,data/raw/LFW/lfw-deepfunneled/lfw-deepfunneled...
1,lfw:Abdullatif_Sener:Abdullatif_Sener_0001,lfw:Abdullatif_Sener,calibration,data/raw/LFW/lfw-deepfunneled/lfw-deepfunneled...
2,lfw:Abdullatif_Sener:Abdullatif_Sener_0002,lfw:Abdullatif_Sener,calibration,data/raw/LFW/lfw-deepfunneled/lfw-deepfunneled...
3,lfw:Adam_Kennedy:Adam_Kennedy_0001,lfw:Adam_Kennedy,calibration,data/raw/LFW/lfw-deepfunneled/lfw-deepfunneled...
4,lfw:Adoor_Gopalakarishnan:Adoor_Gopalakarishna...,lfw:Adoor_Gopalakarishnan,calibration,data/raw/LFW/lfw-deepfunneled/lfw-deepfunneled...


In [3]:
# cell 3 : 얼굴 검출 및 정렬 실행
from research.preprocessing import materialize_aligned_crops

if EXECUTE_STAGE:
    result = materialize_aligned_crops(
        source_manifest,
        project_root=PROJECT_ROOT,
        output_dir=OUTPUT_DIR,
        dataset_id="lfw",
        detector_name=DETECTOR_NAME,
        detection_size=DETECTION_INPUT_SIZE,
        providers=PROVIDERS,
        overwrite=OVERWRITE,
    )
    summary = {
        "aligned_count": result.bundle_metadata["aligned_count"],
        "failed_count": result.bundle_metadata["failed_count"],
        "success_rate": result.bundle_metadata["alignment_success_rate"],
        "npy_shape": list(result.aligned_faces.shape),
        "output_dir": str(result.output_dir),
    }
else:
    summary = {
        "status": "not_executed",
        "reason": "EXECUTE_STAGE=False",
    }
summary


2026-07-25 03:12:32,604 [INFO] research.preprocessing.aligned_crops — InsightFace FaceAnalysis 초기화: model=buffalo_l, providers=('CUDAExecutionProvider', 'CPUExecutionProvider'), det_size=(640, 640)


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'cudnn_conv_algo_search': 'EXHAUSTIVE', 'device_id': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'has_user_compute_stream': '0', 'gpu_external_alloc': '0', 'enable_cuda_graph': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0'}, 'CPUExecutionProvider': {}}
find model: C:\Users\Administrator/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'cudnn_conv_algo_search': 'EXHAUSTIVE', 'device_id': '0', 'cudnn_conv1d_pad_to_nc1d': '0', '

2026-07-25 03:12:34,513 [INFO] research.preprocessing.aligned_crops — 처리 중: 1/13233 (0.0%)


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'cudnn_conv_algo_search': 'EXHAUSTIVE', 'device_id': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'has_user_compute_stream': '0', 'gpu_external_alloc': '0', 'enable_cuda_graph': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0'}, 'CPUExecutionProvider': {}}
find model: C:\Users\Administrator/.insightface\models\buffalo_l\w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)


c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\insightface\utils\transform.py:68: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  P = np.linalg.lstsq(X_homo, Y)[0].T # Affine matrix. 3 x 4
c:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\insightface\utils\face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)
2026-07-25 03:13:04,860 [INFO] research.preprocessing.aligned_crops — 처리 중: 661/13233 (5.0%)
2026-07-25 03:13:29,513 [INFO] research.preprocessing.aligned_crops — 처리 중: 1322/13233 (10.0%)
2026-07-25 03:13:54,815 [INFO] research.

{'aligned_count': 13195,
 'failed_count': 38,
 'success_rate': 0.9971283911433537,
 'npy_shape': [13195, 112, 112, 3],
 'output_dir': 'C:\\ronbun\\data\\interim\\common\\aligned_112\\lfw'}

In [4]:
# cell 4 : 결과 검증 및 요약 보고
import json
import numpy as np

if EXECUTE_STAGE:
    # NPY 형상 검증
    npy_path = OUTPUT_DIR / "aligned_faces.npy"
    faces = np.load(npy_path, mmap_mode="r", allow_pickle=False)
    assert faces.ndim == 4, f"ndim={faces.ndim}"
    assert faces.shape[1:] == (112, 112, 3), f"shape={faces.shape}"
    assert faces.dtype == np.uint8, f"dtype={faces.dtype}"

    # manifest 열 검증
    manifest_path = OUTPUT_DIR / "aligned_manifest.parquet"
    aligned_df = pd.read_parquet(manifest_path)
    required_cols = {
        "sample_id", "aligned_face_index", "aligned_content_sha256",
        "identity_id", "split", "dataset_id",
    }
    missing = sorted(required_cols - set(aligned_df.columns))
    assert not missing, f"누락 열: {missing}"

    # 인덱스 연속성 검증
    indices = aligned_df["aligned_face_index"].to_numpy()
    assert np.array_equal(indices, np.arange(len(aligned_df))), (
        "aligned_face_index가 0부터 연속이 아닙니다."
    )
    assert len(aligned_df) == faces.shape[0], (
        f"manifest행={len(aligned_df)} != NPY행={faces.shape[0]}"
    )

    # bundle manifest
    bundle = json.loads(
        (OUTPUT_DIR / "bundle_manifest.json").read_text(encoding="utf-8")
    )
    logger.info(
        "검증 완료: %d장 정렬, %d장 실패, 성공률 %.2f%%",
        bundle["aligned_count"],
        bundle["failed_count"],
        bundle["alignment_success_rate"] * 100,
    )

    # 실패 보고
    failed_path = OUTPUT_DIR / "failed_samples.parquet"
    if failed_path.is_file():
        failed_df = pd.read_parquet(failed_path)
        logger.info("실패 사유 분포:")
        display(failed_df["alignment_failure_reason"].value_counts())
    else:
        logger.info("실패한 표본이 없습니다.")

    # SUCCESS marker
    assert (OUTPUT_DIR / "_SUCCESS").is_file(), "_SUCCESS marker가 없습니다."
    logger.info("모든 검증 통과")
else:
    logger.info("EXECUTE_STAGE=False — 검증을 건너뜁니다.")


2026-07-25 03:23:10,211 [INFO] aligned_crop_materializer — 검증 완료: 13195장 정렬, 38장 실패, 성공률 99.71%
2026-07-25 03:23:10,216 [INFO] aligned_crop_materializer — 실패 사유 분포:


alignment_failure_reason
no_face_detected    38
Name: count, dtype: int64

2026-07-25 03:23:10,220 [INFO] aligned_crop_materializer — 모든 검증 통과


이 산출물이 생성된 후 `notebooks/lfw/gradcam/00_source_and_model_freeze.ipynb`의
입력 경로가 채워집니다. YAML 설정의 `datasets.lfw.aligned_crops` 블록에서
경로를 읽으므로 노트북에서 수동 입력할 필요가 없습니다.
